# 02 — Legal Unit Chunking

**Purpose:** Group each article with its notes, then create structure-aware retrieval chunks.

Business logic lives in `laborlaw_rag.data`; this notebook only runs and inspects the public API.


## 1. Setup


In [ ]:
from laborlaw_rag.config import Settings
from laborlaw_rag.data import build_legal_units, create_chunks, load_records, save_chunks

settings = Settings.from_env()
MAX_TOKENS = 512
WRITE_CHUNKS = False

## 2. Load Parsed Records


In [ ]:
records = load_records(settings.records_path)
{"record_count": len(records), "records_path": str(settings.records_path)}

## 3. Build Legal Units


In [ ]:
legal_units = build_legal_units(records)
{
    "unit_count": len(legal_units),
    "units_with_notes": sum(unit["has_subarticles"] for unit in legal_units),
    "first_reference": legal_units[0]["article_reference"],
}

## 4. Create and Verify Chunks


In [ ]:
chunks = create_chunks(records, max_tokens=MAX_TOKENS)

assert chunks
assert all(chunk["token_count"] <= MAX_TOKENS for chunk in chunks)
assert all(chunk["source_id"] for chunk in chunks)

{
    "chunk_count": len(chunks),
    "closing_note_chunks": sum(chunk["article_number"] is None for chunk in chunks),
    "largest_chunk_tokens": max(chunk["token_count"] for chunk in chunks),
    "articles_split_across_chunks": len(
        {chunk["article_number"] for chunk in chunks if chunk["total_chunks"] > 1}
    ),
}

## 5. Inspect a Chunk


In [ ]:
{
    "chunk_id": chunks[0]["chunk_id"],
    "article_reference": chunks[0]["article_reference"],
    "subarticle_references": chunks[0]["subarticle_references"],
    "token_count": chunks[0]["token_count"],
    "text_preview": chunks[0]["text"][:300],
}

## 6. Persist the Artifact

Set `WRITE_CHUNKS` to `True` only when intentionally replacing the deployment chunks.


In [ ]:
if WRITE_CHUNKS:
    save_chunks(chunks, settings.chunks_path)

{"write_enabled": WRITE_CHUNKS, "chunks_path": str(settings.chunks_path)}